# CSI Data Preprocessing
Các hàm tiền xử lý dữ liệu CSI (Channel State Information) cho mô hình học máy

## 1. Import Thư Viện

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt

## 2. Hàm Đọc Dữ Liệu CSI CSV

In [ ]:
def load_csi_csv(
	path: str | Path,
	expected_columns: int = 65,
	metadata_columns: int = 2,
	drop_zero_subcarriers: bool = True,
) -> np.ndarray:
	"""Load CSI CSV file and keep only valid rows/subcarriers.
	
	Args:
		path: Đường dẫn đến file CSV
		expected_columns: Số cột dự kiến (mặc định 65)
		metadata_columns: Số cột metadata cần bỏ qua (mặc định 2)
		drop_zero_subcarriers: Bỏ các subcarrier có giá trị 0 (mặc định True)
	
	Returns:
		np.ndarray: Ma trận CSI với shape (n_samples, n_subcarriers)
	"""
	rows: list[list[str]] = []
	source = Path(path)

	with source.open("r", encoding="utf-8") as f:
		for line in f:
			values = line.strip().split(",")
			if len(values) == expected_columns:
				rows.append(values)

	if not rows:
		raise ValueError(f"No valid CSI rows found in {source}")

	data = pd.DataFrame(rows).astype(float)
	csi = data.iloc[:, metadata_columns:].values

	if drop_zero_subcarriers:
		non_zero = np.any(csi != 0, axis=0)
		csi = csi[:, non_zero]

	return csi

In [2]:
def standardize_csi(csi: np.ndarray) -> np.ndarray:
    """Chuẩn hóa Z-score: Làm nổi bật hình dạng biến động của từng subcarrier."""
    mu = np.mean(csi, axis=0)
    sigma = np.std(csi, axis=0)
    return (csi - mu) / (sigma + 1e-8)

## 3. Hàm Căn Chỉnh Kích Thước Feature

In [ ]:
def align_feature_dims(a: np.ndarray, b: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
	"""Align two CSI arrays by truncating both to smallest feature dimension.
	
	Args:
		a: Ma trận CSI thứ nhất
		b: Ma trận CSI thứ hai
	
	Returns:
		tuple: (a_aligned, b_aligned) - hai ma trận đã được căn chỉnh
	"""
	feature_dim = min(a.shape[1], b.shape[1])
	return a[:, :feature_dim], b[:, :feature_dim]

## 4. Hàm Căn Chỉnh Kích Thước Feature Cho Nhiều Mảng

In [3]:
def align_feature_dims_multi(*arrays: np.ndarray) -> tuple[np.ndarray, ...]:
	"""Align multiple CSI arrays by truncating all to the smallest feature dimension.

	Args:
		*arrays: Số lượng bất kỳ ma trận CSI cần căn chỉnh
	
	Returns:
		tuple: Các mảng đã được căn chỉnh theo cùng một chiều thứ hai
	
	Example: a, b, c = align_feature_dims_multi(a, b, c)
	"""
	if not arrays:
		raise ValueError("No arrays provided to align_feature_dims_multi")
	feature_dims = [arr.shape[1] for arr in arrays]
	min_dim = min(feature_dims)
	return tuple(arr[:, :min_dim] for arr in arrays)

## 5. Hàm Lọc Hampel (Loại Bỏ Nhiễu)

In [4]:
def hampel_filter_1d(signal: np.ndarray, window_size: int = 5, n_sigmas: float = 3.0) -> np.ndarray:
	"""Remove outliers in one-dimensional signal using Hampel filter.
	
	Args:
		signal: Tín hiệu 1 chiều
		window_size: Kích thước cửa sổ (mặc định 5)
		n_sigmas: Số sigma để xác định ngưỡng nhiễu (mặc định 3.0)
	
	Returns:
		np.ndarray: Tín hiệu đã được lọc
	"""
	filtered = signal.copy()
	for idx in range(window_size, len(signal) - window_size):
		window = signal[idx - window_size : idx + window_size]
		median = np.median(window)
		mad = np.median(np.abs(window - median))

		if mad == 0:
			continue

		threshold = n_sigmas * 1.4826 * mad
		if abs(signal[idx] - median) > threshold:
			filtered[idx] = median

	return filtered

## 6. Hàm Áp Dụng Lọc Hampel Trên Toàn Bộ CSI

In [5]:
def apply_hampel(csi: np.ndarray, window_size: int = 5, n_sigmas: float = 3.0) -> np.ndarray:
	"""Apply Hampel filter independently on each subcarrier.
	
	Args:
		csi: Ma trận CSI với shape (n_samples, n_subcarriers)
		window_size: Kích thước cửa sổ cho Hampel filter (mặc định 5)
		n_sigmas: Số sigma (mặc định 3.0)
	
	Returns:
		np.ndarray: Ma trận CSI đã được lọc
	"""
	output = np.zeros_like(csi)
	for col in range(csi.shape[1]):
		output[:, col] = hampel_filter_1d(csi[:, col], window_size=window_size, n_sigmas=n_sigmas)
	return output

## 7. Hàm Lọc Butterworth (Low-Pass Filter)

In [6]:
def butterworth_lowpass(csi: np.ndarray, order: int = 4, cutoff: float = 0.1) -> np.ndarray:
	"""Apply low-pass Butterworth filter over time axis.
	
	Args:
		csi: Ma trận CSI với shape (n_samples, n_subcarriers)
		order: Độ của bộ lọc (mặc định 4)
		cutoff: Tần số cắt chuẩn hóa từ 0 đến 1 (mặc định 0.1)
	
	Returns:
		np.ndarray: Ma trận CSI đã được lọc
	"""
	b, a = butter(order, cutoff, btype="low")
	return filtfilt(b, a, csi, axis=0)

## 8. Hàm Chuẩn Hóa Dữ Liệu

In [7]:
def normalize_global(x: np.ndarray, eps: float = 1e-8) -> np.ndarray:
	"""Normalize tensor by global max absolute value.
	
	Args:
		x: Mảng dữ liệu cần chuẩn hóa
		eps: Giá trị epsilon để tránh chia cho 0 (mặc định 1e-8)
	
	Returns:
		np.ndarray: Mảng dữ liệu đã được chuẩn hóa
	"""
	max_abs = np.max(np.abs(x))
	return x / (max_abs + eps)

## 9. Ví Dụ Sử Dụng

In [ ]:
# Ví dụ: Nạp dữ liệu CSI
# csi_data = load_csi_csv('../data/raw/csi_raw.csv')
# print(f"CSI data shape: {csi_data.shape}")

In [ ]:
# Ví dụ: Áp dụng Hampel filter
# csi_filtered = apply_hampel(csi_data, window_size=5, n_sigmas=3.0)
# print(f"Filtered CSI shape: {csi_filtered.shape}")

In [ ]:
# Ví dụ: Áp dụng Butterworth low-pass filter
# csi_smooth = butterworth_lowpass(csi_filtered, order=4, cutoff=0.1)
# print(f"Smoothed CSI shape: {csi_smooth.shape}")

In [ ]:
# Ví dụ: Chuẩn hóa dữ liệu
# csi_normalized = normalize_global(csi_smooth)
# print(f"Normalized CSI - min: {csi_normalized.min():.4f}, max: {csi_normalized.max():.4f}")